# FA-UNIFEWS: Complete Experiment Suite

This notebook runs ALL experiments needed for the paper tables.

**Performance Notes (Apple M3, 16GB RAM):**
- `run_fb.py` patched: `cal_flops()` moved out of training loop (was called every epoch!), early stopping enabled
- CS dataset is heaviest (~476 MB features, 6805 dims) — ~30-60s/run
- Computers: ~30-60s/run (491K edges)
- Small datasets (Cornell, Wisconsin): ~5-10s/run
- **Estimated total: ~45 min for single-seed** (was ~3+ hours before patches)

**What's real vs fabricated in the current paper:**
- ✅ UNIFEWS baseline on Cora, Computers, CS → from `save/*/log.csv`
- ❌ All FA-UNIFEWS numbers → **need to run**
- ❌ Heterophilic datasets (Chameleon, Cornell, Wisconsin) → **need to run**
- ❌ MLP baseline → **need to run**
- ❌ Edge homophily ratio → **need to compute**
- ❌ Ablation study → **need to run**

## Structure
1. **Setup & Data Download**
2. **Quick Test** (single run ~30s)
3. **Main Comparison** (18 runs)
4. **MLP + Dense Baselines** (12 runs)
5. **Ablation Study** (14 runs)
6. **Sparsity Sweep** (48 runs)
7. **Edge Homophily** (computation only)
8. **Results Aggregation**

In [1]:
import os
import subprocess
import json
import csv
import pandas as pd
import numpy as np
from pathlib import Path

import os
os.environ["MallocStackLogging"] = "0"

# Base paths
PROJECT_ROOT = Path("/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2")
UNIFEWS_DIR = PROJECT_ROOT / "Unifews"
CONFIG_DIR = UNIFEWS_DIR / "config"
DATA_DIR = UNIFEWS_DIR / "data"
SAVE_DIR = UNIFEWS_DIR / "save"

os.chdir(UNIFEWS_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"Available datasets: {[d.name for d in DATA_DIR.iterdir() if d.is_dir()]}")
print(f"Available configs: {sorted([f.stem for f in CONFIG_DIR.glob('*.json')])}")

Working directory: /Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews
Data directory exists: True
Available datasets: ['cornell', 'computers', 'chameleon', 'cs', 'citeseer', 'wisconsin', 'cora', 'texas', 'pubmed', 'processed', 'raw']
Available configs: ['arxiv', 'arxiv_mb', 'chameleon', 'citeseer', 'citeseer_lp', 'citeseer_mb', 'computers', 'computers_appnp_table', 'computers_appnp_table_thr', 'computers_lp', 'computers_mb', 'computers_sgc_table', 'computers_sgc_table_thr', 'cora', 'cora_appnp_table', 'cora_appnp_table_thr', 'cora_lp', 'cora_mb', 'cora_sgc_table', 'cora_sgc_table_thr', 'cornell', 'cs', 'cs_appnp_table', 'cs_appnp_table_thr', 'cs_lp', 'cs_mb', 'cs_sgc_table', 'cs_sgc_table_thr', 'gencat-2.1-0.1', 'ogbl-collab_lp', 'papers_mb', 'physics', 'products_mb', 'pubmed', 'pubmed_lp', 'pubmed_mb', 'texas', 'wisconsin']


## 1. Download & Prepare Heterophilic Datasets

The heterophilic datasets (Chameleon, Cornell, Wisconsin, Texas) are NOT in the data/ folder.
We need to download them from PyG and convert to the format expected by the code.

In [2]:
"""
Download heterophilic datasets using PyTorch Geometric and convert
to the format expected by Unifews (adj.npz, feats.npy, labels.npz).
"""
import torch
import scipy.sparse as sp

def download_and_convert_pyg_dataset(dataset_name, data_dir):
    """Download a PyG dataset and save in Unifews format."""
    from torch_geometric.datasets import WikipediaNetwork, WebKB, Planetoid
    
    save_path = data_dir / dataset_name
    if save_path.exists() and (save_path / "adj.npz").exists():
        print(f"  {dataset_name}: already exists, skipping download")
        return
    
    save_path.mkdir(parents=True, exist_ok=True)
    tmp_root = data_dir / "raw"
    tmp_root.mkdir(exist_ok=True)
    
    # Load dataset
    if dataset_name in ['chameleon', 'squirrel']:
        dataset = WikipediaNetwork(root=str(tmp_root), name=dataset_name, transform=None)
    elif dataset_name in ['cornell', 'texas', 'wisconsin']:
        dataset = WebKB(root=str(tmp_root), name=dataset_name, transform=None)
    elif dataset_name in ['citeseer', 'pubmed']:
        dataset = Planetoid(root=str(tmp_root), name=dataset_name, transform=None)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    
    graph = dataset[0]
    n = graph.num_nodes
    
    # Build adjacency (undirected, no self-loops)
    row = graph.edge_index[0].numpy()
    col = graph.edge_index[1].numpy()
    # Make undirected
    row_full = np.concatenate([row, col])
    col_full = np.concatenate([col, row])
    data = np.ones(len(row_full), dtype=np.int8)
    adj = sp.csr_matrix((data, (row_full, col_full)), shape=(n, n))
    adj.setdiag(0)
    adj.eliminate_zeros()
    adj.data = np.ones(adj.nnz, dtype=np.int8)
    
    # Features
    feats = graph.x.numpy()
    
    # Labels
    labels = graph.y.numpy().flatten()
    
    # Train/val/test split (use first split if available, else random 50/25/25)
    if hasattr(graph, 'train_mask') and graph.train_mask is not None:
        if graph.train_mask.dim() > 1:
            # Multiple splits, use first
            idx_train = torch.where(graph.train_mask[:, 0])[0].numpy()
            idx_val = torch.where(graph.val_mask[:, 0])[0].numpy()
            idx_test = torch.where(graph.test_mask[:, 0])[0].numpy()
        else:
            idx_train = torch.where(graph.train_mask)[0].numpy()
            idx_val = torch.where(graph.val_mask)[0].numpy()
            idx_test = torch.where(graph.test_mask)[0].numpy()
    else:
        # Random split: 50/25/25
        perm = np.random.RandomState(42).permutation(n)
        n_train = int(0.5 * n)
        n_val = int(0.25 * n)
        idx_train = perm[:n_train]
        idx_val = perm[n_train:n_train+n_val]
        idx_test = perm[n_train+n_val:]
    
    # Save
    sp.save_npz(str(save_path / "adj.npz"), adj.tocsc())
    np.save(str(save_path / "feats.npy"), feats)
    np.savez(str(save_path / "labels.npz"),
             labels=labels,
             idx_train=idx_train,
             idx_val=idx_val,
             idx_test=idx_test)
    
    # Degree
    degree = np.array(adj.sum(axis=1)).flatten()
    sp.save_npz(str(save_path / "degree.npz"), 
                sp.csr_matrix(degree.reshape(-1, 1)))
    
    # Edge homophily
    rows, cols = adj.nonzero()
    same_label = (labels[rows] == labels[cols]).sum()
    homophily = same_label / len(rows) if len(rows) > 0 else 0
    
    print(f"  {dataset_name}: n={n}, m={adj.nnz}, f={feats.shape[1]}, "
          f"classes={len(np.unique(labels))}, homophily={homophily:.3f}")
    print(f"    split: train={len(idx_train)}, val={len(idx_val)}, test={len(idx_test)}")

# Download all missing datasets
print("=== Downloading missing datasets ===")
for ds_name in ['chameleon', 'cornell', 'wisconsin', 'texas', 'citeseer', 'pubmed']:
    try:
        download_and_convert_pyg_dataset(ds_name, DATA_DIR)
    except Exception as e:
        print(f"  {ds_name}: FAILED - {e}")

print("\n=== All datasets ===")
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir() and (d / "adj.npz").exists():
        adj = sp.load_npz(str(d / "adj.npz"))
        labels = np.load(str(d / "labels.npz"), allow_pickle=True)
        print(f"  {d.name}: n={adj.shape[0]}, m={adj.nnz}, labels={labels['labels'].shape}")

=== Downloading missing datasets ===
  chameleon: already exists, skipping download
  cornell: already exists, skipping download
  wisconsin: already exists, skipping download
  texas: already exists, skipping download
  citeseer: already exists, skipping download
  pubmed: already exists, skipping download

=== All datasets ===
  chameleon: n=2277, m=62742, labels=(2277,)
  citeseer: n=3327, m=9104, labels=(3327,)
  computers: n=13752, m=491722, labels=(13752,)
  cora: n=2485, m=12623, labels=(2485,)
  cornell: n=183, m=554, labels=(183,)
  cs: n=18333, m=163788, labels=(18333,)
  pubmed: n=19717, m=88648, labels=(19717,)
  texas: n=183, m=558, labels=(183,)
  wisconsin: n=251, m=900, labels=(251,)


## 2. Helper: Run experiment & collect results

In [3]:
def run_experiment(config, algo="gcn_thr", seed=42, thr_a=0.5, thr_w=0.5, 
                   fa_alpha=1.0, device=0, extra_args=None):
    """
    Run a single experiment via run_fb.py and return the result dict.
    
    fa_alpha meanings:
      1.0  → Mode 1: Vanilla UNIFEWS
      0.5  → Mode 2: Static FA-UNIFEWS (alpha=0.5)
     -1.0  → Mode 3: Adaptive FA-UNIFEWS
    """
    import re
    cmd = [
        "python", "run_fb.py",
        "-f", str(seed),
        "-c", config,
        "-m", algo,
        "-a", str(thr_a),
        "-w", str(thr_w),
        "-v", str(device),
        "--fa_alpha", str(fa_alpha),
    ]
    if extra_args:
        cmd.extend(extra_args)
    
    cmd_str = " ".join(cmd)
    print(f"  Running: {cmd_str}")
    
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        cwd=str(UNIFEWS_DIR), timeout=600
    )
    
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr[-500:]}")
        return {"config": config, "algo": algo, "seed": seed,
                "thr_a": thr_a, "thr_w": thr_w, "fa_alpha": fa_alpha,
                "acc": None, "error": result.stderr[-500:]}
    
    # Parse output for accuracy
    output = result.stdout + result.stderr
    acc = None
    numel_a = None
    numel_w = None
    time_train = None
    time_test = None
    
    for line in output.split('\n'):
        # Parse: [Test] best acc: 0.87942
        if '[test]' in line.lower() and 'best acc' in line.lower():
            m = re.search(r'best acc:\s*([0-9.]+)', line, re.IGNORECASE)
            if m:
                acc = float(m.group(1))
        
        # Parse: [Test]  time: 0.1132 s, MACs: 1.0655 G, Num adj: 19.654 k, Num weight: 420.819 k
        if '[test]' in line.lower() and 'num adj' in line.lower():
            m_na = re.search(r'Num adj:\s*([0-9.]+)', line)
            m_nw = re.search(r'Num weight:\s*([0-9.]+)', line)
            m_tt = re.search(r'time:\s*([0-9.]+)', line)
            if m_na: numel_a = float(m_na.group(1))
            if m_nw: numel_w = float(m_nw.group(1))
            if m_tt: time_test = float(m_tt.group(1))
        
        # Parse: [Train] time: 18.0879 s
        if '[train]' in line.lower() and 'time:' in line.lower():
            m_tr = re.search(r'time:\s*([0-9.]+)', line)
            if m_tr: time_train = float(m_tr.group(1))
        
        # Fallback: parse CSV line (last line usually)
        # Format: cora ,gcn_thr ,42,7.00e-01,5.00e-01,0.87942,...
        if acc is None and ',' in line:
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 14:
                try:
                    maybe_acc = float(parts[5])
                    if 0 < maybe_acc < 1:
                        acc = maybe_acc
                        numel_a = float(parts[12])
                        numel_w = float(parts[13])
                except (ValueError, IndexError):
                    pass
    
    return {
        "config": config,
        "algo": algo,
        "seed": seed,
        "thr_a": thr_a,
        "thr_w": thr_w,
        "fa_alpha": fa_alpha,
        "acc": acc,
        "numel_a": numel_a,
        "numel_w": numel_w,
        "time_train": time_train,
        "time_test": time_test,
        "stdout": result.stdout[-1000:],
        "stderr": result.stderr[-500:],
    }


def run_experiment_batch(experiments, desc=""):
    """Run a batch of experiments and collect results."""
    results = []
    total = len(experiments)
    print(f"\n{'='*60}")
    print(f"Running {total} experiments: {desc}")
    print(f"{'='*60}")
    
    for i, exp in enumerate(experiments):
        print(f"\n[{i+1}/{total}] {exp.get('config', '?')} | "
              f"algo={exp.get('algo', '?')} | fa_alpha={exp.get('fa_alpha', '?')} | "
              f"thr_a={exp.get('thr_a', '?')}")
        try:
            r = run_experiment(**exp)
            results.append(r)
            if r and r.get('acc'):
                print(f"  → Accuracy: {r['acc']:.4f}")
            else:
                print(f"  → Could not parse accuracy: {r.get('error', 'unknown')}")
        except subprocess.TimeoutExpired:
            print(f"  → TIMEOUT (>600s)")
            results.append({**exp, "acc": None, "error": "timeout"})
        except Exception as e:
            print(f"  → EXCEPTION: {e}")
            results.append({**exp, "acc": None, "error": str(e)})
    
    return results

print("Helper functions defined.")

Helper functions defined.


## 3. Experiment 1: Main Comparison Table (Table 3 in paper)

Run UNIFEWS (fa_alpha=1.0) vs FA-UNIFEWS_static (fa_alpha=0.5) vs FA-UNIFEWS_adapt (fa_alpha=-1.0)
on all available datasets with τ_a=0.7, δ_w=0.5.

In [5]:
# ============================================================
# EXPERIMENT 1: Main comparison - 3 methods × 6 datasets × 1 seed
# ============================================================
# Split into LIGHT (fast) and HEAVY (slow) datasets so you can run incrementally

DATASETS_LIGHT = ["cora", "cornell", "wisconsin", "chameleon"]  # ~5-30s each
DATASETS_HEAVY = ["computers", "cs"]                             # ~30-60s each
DATASETS = DATASETS_LIGHT + DATASETS_HEAVY

FA_MODES = {
    "UNIFEWS":           1.0,   # Mode 1: vanilla
    "FA-UNI_static":     0.5,   # Mode 2: static alpha=0.5  
    "FA-UNI_adapt":     -1.0,   # Mode 3: adaptive
}
THR_A = 0.7
THR_W = 0.5
SEED = 42

# Light datasets first (fast feedback)
experiments_main_light = []
for dataset in DATASETS_LIGHT:
    for mode_name, fa_alpha in FA_MODES.items():
        experiments_main_light.append({
            "config": dataset, "algo": "gcn_thr", "seed": SEED,
            "thr_a": THR_A, "thr_w": THR_W, "fa_alpha": fa_alpha,
        })

# Heavy datasets separate
experiments_main_heavy = []
for dataset in DATASETS_HEAVY:
    for mode_name, fa_alpha in FA_MODES.items():
        experiments_main_heavy.append({
            "config": dataset, "algo": "gcn_thr", "seed": SEED,
            "thr_a": THR_A, "thr_w": THR_W, "fa_alpha": fa_alpha,
        })

experiments_main = experiments_main_light + experiments_main_heavy

print(f"Light experiments: {len(experiments_main_light)} (~5 min)")
print(f"Heavy experiments: {len(experiments_main_heavy)} (~5 min)")
print(f"Total: {len(experiments_main)} experiments")
print("\nNext cell runs LIGHT first, then HEAVY.")

Light experiments: 12 (~5 min)
Heavy experiments: 6 (~5 min)
Total: 18 experiments

Next cell runs LIGHT first, then HEAVY.


In [ ]:
# Run LIGHT datasets first (~5 min) for quick feedback
results_main_light = run_experiment_batch(experiments_main_light, desc="Main Comparison - LIGHT datasets")

# Then run HEAVY datasets (~5 min)
results_main_heavy = run_experiment_batch(experiments_main_heavy, desc="Main Comparison - HEAVY datasets")

results_main = results_main_light + results_main_heavy

# Quick summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for r in results_main:
    acc_str = f"{r['acc']*100:.2f}%" if r.get('acc') else "FAILED"
    print(f"  {r['config']:12s} | fa_alpha={r['fa_alpha']:5.1f} | {acc_str}")


Running 12 experiments: Main Comparison - LIGHT datasets

[1/12] cora | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 1.0
  → Accuracy: 0.8567

[2/12] cora | algo=gcn_thr | fa_alpha=0.5 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 0.5
  → Accuracy: 0.8679

[3/12] cora | algo=gcn_thr | fa_alpha=-1.0 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha -1.0
  → Accuracy: 0.8712

[4/12] cornell | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cornell -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 1.0
  → Accuracy: 0.6271

[5/12] cornell | algo=gcn_thr | fa_alpha=0.5 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cornell -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 0.5
  → Accuracy: 0.7119

[6/12] cornell | algo=gcn_thr | fa_alpha=-1.0 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cornell -m gcn_th

## 4. Experiment 2: MLP Baseline

Run MLP (no graph structure) on all datasets for Computational Shock reference line.

In [ ]:
# ============================================================
# EXPERIMENT 2: MLP baselines
# ============================================================
experiments_mlp = []
for dataset in DATASETS:
    experiments_mlp.append({
        "config": dataset,
        "algo": "mlp",     # MLP model, no graph operations
        "seed": SEED,
        "thr_a": 0.0,
        "thr_w": 0.0,
        "fa_alpha": 1.0,   # doesn't matter for MLP
    })

print(f"Total MLP experiments: {len(experiments_mlp)}")

Total MLP experiments: 6


In [ ]:
results_mlp = run_experiment_batch(experiments_mlp, desc="MLP Baselines")


Running 6 experiments: MLP Baselines

[1/6] cora | algo=mlp | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c cora -m mlp -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  → Accuracy: 0.7552

[2/6] cs | algo=mlp | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c cs -m mlp -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  → Accuracy: 0.9496

[3/6] computers | algo=mlp | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c computers -m mlp -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  → Accuracy: 0.8508

[4/6] chameleon | algo=mlp | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c chameleon -m mlp -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0


KeyboardInterrupt: 

## 5. Experiment 3: Dense GCN Baseline (no pruning)

In [ ]:
# ============================================================
# EXPERIMENT 3: Dense GCN (thr_a=0, thr_w=0, fa_alpha=1.0)
# ============================================================
experiments_dense = []
for dataset in DATASETS:
    experiments_dense.append({
        "config": dataset,
        "algo": "gcn_thr",
        "seed": SEED,
        "thr_a": 0.0,
        "thr_w": 0.0,
        "fa_alpha": 1.0,
    })

print(f"Total dense GCN experiments: {len(experiments_dense)}")

Total dense GCN experiments: 6


In [ ]:
results_dense = run_experiment_batch(experiments_dense, desc="Dense GCN Baselines")


Running 6 experiments: Dense GCN Baselines

[1/6] cora | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  ERROR: Traceback (most recent call last):
  File "/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews/run_fb.py", line 7, in <module>
    import ptflops
ModuleNotFoundError: No module named 'ptflops'

  → Could not parse accuracy

[2/6] cs | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c cs -m gcn_thr -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  ERROR: Traceback (most recent call last):
  File "/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews/run_fb.py", line 7, in <module>
    import ptflops
ModuleNotFoundError: No module named 'ptflops'

  → Could not parse accuracy

[3/6] computers | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c computers -m gcn_thr -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  ERROR: Traceback (most recent cal

## 6. Experiment 4: Ablation Study (Table 6 in paper)

Test each component on Cora and Chameleon:
- Full FA-UNIFEWS_adapt
- w/o cosine (alpha=1.0, = UNIFEWS) 
- w/o adaptive gate (static alpha=0.5)
- Static alpha sweep: 0.1, 0.3, 0.5, 0.7, 0.9

In [ ]:
# ============================================================
# EXPERIMENT 4: Ablation
# ============================================================
ABLATION_DATASETS = ["cora", "chameleon"]
ALPHA_SWEEP = [0.1, 0.3, 0.5, 0.7, 0.9]

experiments_ablation = []
for dataset in ABLATION_DATASETS:
    # Full adaptive
    experiments_ablation.append({
        "config": dataset, "algo": "gcn_thr", "seed": SEED,
        "thr_a": THR_A, "thr_w": THR_W, "fa_alpha": -1.0,
    })
    # w/o cosine (= UNIFEWS)
    experiments_ablation.append({
        "config": dataset, "algo": "gcn_thr", "seed": SEED,
        "thr_a": THR_A, "thr_w": THR_W, "fa_alpha": 1.0,
    })
    # Alpha sweep (static mode)
    for alpha in ALPHA_SWEEP:
        experiments_ablation.append({
            "config": dataset, "algo": "gcn_thr", "seed": SEED,
            "thr_a": THR_A, "thr_w": THR_W, "fa_alpha": alpha,
        })

print(f"Total ablation experiments: {len(experiments_ablation)}")

Total ablation experiments: 14


In [ ]:
results_ablation = run_experiment_batch(experiments_ablation, desc="Ablation Study (Table 6)")


Running 14 experiments: Ablation Study (Table 6)

[1/14] cora | algo=gcn_thr | fa_alpha=-1.0 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha -1.0
  ERROR: Traceback (most recent call last):
  File "/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews/run_fb.py", line 7, in <module>
    import ptflops
ModuleNotFoundError: No module named 'ptflops'

  → Could not parse accuracy

[2/14] cora | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 1.0
  ERROR: Traceback (most recent call last):
  File "/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews/run_fb.py", line 7, in <module>
    import ptflops
ModuleNotFoundError: No module named 'ptflops'

  → Could not parse accuracy

[3/14] cora | algo=gcn_thr | fa_alpha=0.1 | thr_a=0.7
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 0.1
  ERROR: Traceback (most recen

## 7. Experiment 5: Sparsity Sweep (Table 7 in paper)

Sweep τ_a values to generate sparsity-accuracy curves.

In [ ]:
# ============================================================
# EXPERIMENT 5: Sparsity sweep
# ============================================================
THR_A_VALUES = [0.0, 0.2, 0.4, 0.8, 1.0, 1.5, 2.0, 3.0]
SWEEP_DATASETS = ["cora", "computers", "cs"]

experiments_sweep = []
for dataset in SWEEP_DATASETS:
    for thr_a in THR_A_VALUES:
        # UNIFEWS baseline
        experiments_sweep.append({
            "config": dataset, "algo": "gcn_thr", "seed": SEED,
            "thr_a": thr_a, "thr_w": 0.0, "fa_alpha": 1.0,
        })
        # FA-UNIFEWS static
        experiments_sweep.append({
            "config": dataset, "algo": "gcn_thr", "seed": SEED,
            "thr_a": thr_a, "thr_w": 0.0, "fa_alpha": 0.5,
        })

print(f"Total sweep experiments: {len(experiments_sweep)}")

Total sweep experiments: 48


In [ ]:
results_sweep = run_experiment_batch(experiments_sweep, desc="Sparsity Sweep (Table 7)")


Running 48 experiments: Sparsity Sweep (Table 7)

[1/48] cora | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.0
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.0 -w 0.0 -v 0 --fa_alpha 1.0
  ERROR: Traceback (most recent call last):
  File "/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews/run_fb.py", line 7, in <module>
    import ptflops
ModuleNotFoundError: No module named 'ptflops'

  → Could not parse accuracy

[2/48] cora | algo=gcn_thr | fa_alpha=0.5 | thr_a=0.0
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.0 -w 0.0 -v 0 --fa_alpha 0.5
  ERROR: Traceback (most recent call last):
  File "/Users/nhantt/Downloads/MS_HCMUS/Research/ConfA/Project 2/Unifews/run_fb.py", line 7, in <module>
    import ptflops
ModuleNotFoundError: No module named 'ptflops'

  → Could not parse accuracy

[3/48] cora | algo=gcn_thr | fa_alpha=1.0 | thr_a=0.2
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.2 -w 0.0 -v 0 --fa_alpha 1.0
  ERROR: Traceback (most recent 

## 8. Compute Edge Homophily Ratio (Table 4 in paper)

This computes the edge homophily ratio of the pruned graphs.
We need to run the model, capture the pruned edge set, and measure homophily.

In [ ]:
"""
Compute edge homophily ratio before and after pruning.
This requires importing the model internals.
"""
import sys
sys.path.insert(0, str(UNIFEWS_DIR))

import scipy.sparse as sp

def compute_edge_homophily(dataset_name):
    """Compute original edge homophily for a dataset."""
    data_path = DATA_DIR / dataset_name
    adj = sp.load_npz(str(data_path / "adj.npz"))
    labels_data = np.load(str(data_path / "labels.npz"), allow_pickle=True)
    labels = labels_data["labels"]
    
    rows, cols = adj.nonzero()
    if len(rows) == 0:
        return 0.0
    same_label = (labels[rows] == labels[cols]).sum()
    homophily = same_label / len(rows)
    return homophily

print("=== Original Edge Homophily ===")
for ds in ["cora", "computers", "cs", "chameleon", "cornell", "wisconsin"]:
    try:
        h = compute_edge_homophily(ds)
        print(f"  {ds}: H = {h:.4f}")
    except Exception as e:
        print(f"  {ds}: ERROR - {e}")

=== Original Edge Homophily ===
  cora: H = 0.8041
  computers: H = 0.7772
  cs: H = 0.8081
  chameleon: H = 0.2299
  cornell: H = 0.1227
  wisconsin: H = 0.1778


## 9. Aggregate All Results & Generate Paper Tables

In [ ]:
def results_to_df(results, label=""):
    """Convert results list to DataFrame."""
    rows = []
    for r in results:
        if r is None:
            continue
        rows.append({
            "config": r.get("config", ""),
            "algo": r.get("algo", ""),
            "fa_alpha": r.get("fa_alpha", ""),
            "thr_a": r.get("thr_a", ""),
            "thr_w": r.get("thr_w", ""),
            "acc": r.get("acc", None),
            "numel_a": r.get("numel_a", None),
            "numel_w": r.get("numel_w", None),
            "time_train": r.get("time_train", None),
            "time_test": r.get("time_test", None),
            "label": label,
        })
    return pd.DataFrame(rows)

# Combine all results
g = globals()
all_results = []
for name, var_name in [
    ("main", "results_main"),
    ("mlp", "results_mlp"),
    ("dense", "results_dense"),
    ("ablation", "results_ablation"),
    ("sweep", "results_sweep"),
]:
    res_list = g.get(var_name, [])
    if res_list:
        df = results_to_df(res_list, label=name)
        if len(df) > 0:
            all_results.append(df)

if all_results:
    df_all = pd.concat(all_results, ignore_index=True)
    print(f"Total results: {len(df_all)}")
    print(df_all.to_string())
    
    # Save to CSV
    output_path = SAVE_DIR / "all_fa_unifews_results.csv"
    df_all.to_csv(output_path, index=False)
    print(f"\nSaved to: {output_path}")
else:
    print("No results collected yet. Run experiment cells first.")

No results collected yet. Run experiment cells first.


In [ ]:
# ============================================================
# Generate LaTeX table for paper (Table 3: Main Results)
# ============================================================
def generate_main_table_latex(df):
    """Generate LaTeX code for the main comparison table."""
    datasets = ["cora", "cs", "computers", "chameleon", "cornell", "wisconsin"]
    methods = {
        "Dense GCN": {"label": "dense", "fa_alpha": 1.0, "thr_a": 0.0},
        "MLP": {"label": "mlp", "fa_alpha": 1.0, "thr_a": 0.0},
        "UNIFEWS": {"label": "main", "fa_alpha": 1.0, "thr_a": 0.7},
        "FA-UNI$_\\text{static}$": {"label": "main", "fa_alpha": 0.5, "thr_a": 0.7},
        "FA-UNI$_\\text{adapt}$": {"label": "main", "fa_alpha": -1.0, "thr_a": 0.7},
    }
    
    print("% Auto-generated LaTeX table")
    print("\\begin{tabular}{lcccccc}")
    print("\\toprule")
    print(" & \\textbf{Cora} & \\textbf{CS} & \\textbf{Comp.} & "
          "\\textbf{Cham.} & \\textbf{Cornell} & \\textbf{Wisc.} \\\\")
    print("\\midrule")
    
    for method_name, criteria in methods.items():
        row = [method_name]
        for ds in datasets:
            mask = (df["config"] == ds) & (df["label"] == criteria["label"])
            if criteria["label"] == "main":
                mask = mask & (df["fa_alpha"] == criteria["fa_alpha"])
            subset = df[mask]
            if len(subset) > 0 and subset.iloc[0]["acc"] is not None:
                acc = subset.iloc[0]["acc"] * 100 if subset.iloc[0]["acc"] < 1 else subset.iloc[0]["acc"]
                row.append(f"{acc:.2f}")
            else:
                row.append("--")
        print(" & ".join(row) + " \\\\")
    
    print("\\bottomrule")
    print("\\end{tabular}")

if 'df_all' in dir():
    generate_main_table_latex(df_all)
else:
    print("Run experiments first to generate tables.")

Run experiments first to generate tables.


## 10. Quick Test: Verify a single run works

Use this to test that the pipeline works before running the full batch.

In [4]:
# Quick single-run test
print("Testing single experiment on Cora...")
test_result = run_experiment(
    config="cora",
    algo="gcn_thr",
    seed=42,
    thr_a=0.7,
    thr_w=0.5,
    fa_alpha=0.5,  # Static FA-UNIFEWS
    device=0,
)

if test_result:
    print(f"\nResult: acc={test_result['acc']}")
    print(f"Stdout (last 500 chars):\n{test_result['stdout'][-500:]}")
    if test_result['stderr']:
        print(f"Stderr (last 300 chars):\n{test_result['stderr'][-300:]}")
else:
    print("Test failed!")

Testing single experiment on Cora...
  Running: python run_fb.py -f 42 -c cora -m gcn_thr -a 0.7 -w 0.5 -v 0 --fa_alpha 0.5

Result: acc=0.86795
Stdout (last 500 chars):
9 | train loss:0.1231, val acc:0.8599, time:2.4142
[Val] best acc: 0.86795 (epoch: 8/29), [Test] best acc: 0.87942
[Train] time: 2.4142 s (avg: 83.2 ms), MACs: 0.979 G (avg: 1.0 G)
[Test]  time: 0.0585 s, MACs: 1.0654 G, Num adj: 19.574 k, Num weight: 420.819 k
      Data|     Model|  Seed|     ThA|     ThW|    Acc|  Cn|  EP|  Ttrain|  Ctrain|   Ttest|   CTest|  NumelA|  NumelW
cora      ,gcn_thr   ,    42,7.00e-01,5.00e-01,0.87942,   8,  29,  2.4142,   0.979,  0.0585,  1.0654,  19.574, 420.819



## 11. Existing Data: Parse CSV logs already collected

These are the REAL results from previous runs (UNIFEWS baseline only).

In [ ]:
# ============================================================
# Parse existing CSV logs (REAL data from previous experiments)
# ============================================================
def parse_existing_logs():
    """Read all existing CSV log files and compile into a DataFrame."""
    all_rows = []
    
    for dataset_dir in SAVE_DIR.iterdir():
        if not dataset_dir.is_dir():
            continue
        for csv_file in dataset_dir.glob("log*.csv"):
            try:
                with open(csv_file, 'r') as f:
                    content = f.read().strip()
                
                # Handle multi-format CSVs
                lines = content.split('\n')
                if not lines:
                    continue
                    
                header = lines[0].strip()
                # Determine format
                if 'source_file' in header.lower() or 'sourcefile' in header.lower():
                    # Aggregated format
                    for line in lines[1:]:
                        parts = [p.strip() for p in line.split(',')]
                        if len(parts) >= 8:
                            try:
                                row = {
                                    'source': csv_file.name,
                                    'dataset': dataset_dir.name,
                                    'thr_a': float(parts[-8] if 'ThA' not in header else parts[header.split(',').index('ThA') if 'ThA' in header else 5]),
                                    'thr_w': float(parts[-7] if 'ThW' not in header else parts[header.split(',').index('ThW') if 'ThW' in header else 6]),
                                    'acc': float(parts[-6] if 'Acc' not in header else parts[header.split(',').index('Acc') if 'Acc' in header else 7]),
                                }
                                all_rows.append(row)
                            except (ValueError, IndexError):
                                pass
                else:
                    # Standard format
                    for line in lines[1:]:
                        parts = [p.strip() for p in line.split(',')]
                        if len(parts) >= 14:
                            try:
                                row = {
                                    'source': csv_file.name,
                                    'dataset': parts[0].strip(),
                                    'model': parts[1].strip(),
                                    'seed': int(parts[2]),
                                    'thr_a': float(parts[3]),
                                    'thr_w': float(parts[4]),
                                    'acc': float(parts[5]),
                                    'epochs': int(parts[7]),
                                    'train_time': float(parts[8]),
                                    'train_macs': float(parts[9]),
                                    'test_time': float(parts[10]),
                                    'test_macs': float(parts[11]),
                                    'numel_a': float(parts[12]),
                                    'numel_w': float(parts[13]),
                                }
                                all_rows.append(row)
                            except (ValueError, IndexError):
                                pass
            except Exception as e:
                print(f"  Error reading {csv_file}: {e}")
    
    return pd.DataFrame(all_rows)

df_existing = parse_existing_logs()
print(f"Parsed {len(df_existing)} existing experiment records")
print(f"\nDatasets: {df_existing['dataset'].unique() if 'dataset' in df_existing.columns else 'N/A'}")

if len(df_existing) > 0:
    # Show summary
    print("\n=== Existing Results Summary ===")
    for ds in df_existing['dataset'].unique():
        subset = df_existing[df_existing['dataset'] == ds]
        if 'acc' in subset.columns:
            print(f"\n{ds}: {len(subset)} runs")
            print(f"  Acc range: {subset['acc'].min():.4f} - {subset['acc'].max():.4f}")
            if 'thr_a' in subset.columns:
                print(f"  ThA range: {subset['thr_a'].min():.2f} - {subset['thr_a'].max():.2f}")

Parsed 128 existing experiment records

Datasets: ['computers' 'cs' 'cora']

=== Existing Results Summary ===

computers: 38 runs
  Acc range: 0.6021 - 0.9110
  ThA range: 0.00 - 3.00

cs: 59 runs
  Acc range: 0.2626 - 0.9455
  ThA range: 0.00 - 3.00

cora: 31 runs
  Acc range: 0.8537 - 0.8955
  ThA range: 0.00 - 3.00


## 12. Multi-seed runs for mean ± std (for final paper)

For the final paper, we need 10 seeds per experiment. 
Run this AFTER validating single-seed results above.

In [ ]:
# ============================================================
# EXPERIMENT: Multi-seed for main table (10 seeds)
# ⚠️ This is EXPENSIVE: 3 methods × 6 datasets × 10 seeds = 180 runs
# Only run when you're ready for final numbers
# ============================================================
SEEDS = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

experiments_multiseed = []
for dataset in DATASETS:
    for mode_name, fa_alpha in FA_MODES.items():
        for seed in SEEDS:
            experiments_multiseed.append({
                "config": dataset,
                "algo": "gcn_thr",
                "seed": seed,
                "thr_a": THR_A,
                "thr_w": THR_W,
                "fa_alpha": fa_alpha,
            })

print(f"Total multi-seed experiments: {len(experiments_multiseed)}")
print("⚠️ This will take ~90 minutes. Only run when ready for final paper numbers.")
print("Uncomment the line below to execute:")
print("# results_multiseed = run_experiment_batch(experiments_multiseed, 'Multi-seed Main')")

Total multi-seed experiments: 180
⚠️ This will take ~90 minutes. Only run when ready for final paper numbers.
Uncomment the line below to execute:
# results_multiseed = run_experiment_batch(experiments_multiseed, 'Multi-seed Main')
